In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Convolutions for Images

that section derived the convolutional layer on paper, from two
requirements on a fully connected layer: that a shifted input produce a
shifted output, and that each output depend only on a small neighborhood.
The derivation ended with a formula. What does that formula compute on an
actual image, and can a network find good weights for it by itself?

Finding the vertical edges in a grayscale image, the places where a bright
region meets a dark one, is about the simplest instance of both questions.
Picture a small white image with a black vertical band down its middle. If we
subtract from each pixel its right-hand neighbor, the difference is zero
inside flat regions and nonzero where the brightness changes, positive where
white turns to black and negative where black turns to white. An edge can
occur in any row and any column, so the same detector, the two weights
$(1, -1)$, must be applied at every position. Sliding it over the image gives
one output for every position at which it fits, and these outputs form a map
of the edges. This is the operation that a convolutional layer computes: a
small array of weights, the kernel, is compared with every window of the
image, and each comparison yields one output.

Designing a detector by hand works for vertical edges. A network, however,
needs thousands of detectors, and nobody knows in advance which patterns they
should match. We therefore define the operation, implement it, apply it to
the edge example, and then let gradient descent find the kernel $(1, -1)$
from the image and its edge map alone. Two shorter topics follow: how the
operation is computed efficiently, and how much of the image a unit in a
stack of such layers sees.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Cross-Correlation

### The Operation

The edge detector uses a single kernel of shape $1\times 2$. Let us first
state the operation for any input and any kernel, omitting channels for now:
the input $\mathbf{X}$ is a matrix of height $n_\textrm{h}$ and width
$n_\textrm{w}$, and the kernel $\mathbf{K}$ is a
matrix of height $k_\textrm{h}$ and width $k_\textrm{w}$. The kernel's shape
is also the shape of the *window*, the part of the input that the kernel
covers at one position. Place the window with its upper-left corner at input
position $(i, j)$, multiply the input elements under it elementwise with the
kernel, and sum. The result is the output at position $(i, j)$:

$$[\mathbf{Y}]_{i, j} = \sum_{a=0}^{k_\textrm{h}-1} \sum_{b=0}^{k_\textrm{w}-1} [\mathbf{K}]_{a, b}\, [\mathbf{X}]_{i+a, j+b}.$$

This is the *two-dimensional cross-correlation* of $\mathbf{X}$ with
$\mathbf{K}$, which we write $\mathbf{Y} = \mathbf{X} \star \mathbf{K}$.
Indices start at 0 here, as in code. The layer of
that section indexed the window by offsets from $-\Delta$ to
$\Delta$ around its center instead. The two conventions differ only by a
shift of the output by $\Delta$ positions, but the corner is the more
convenient anchor: it also serves kernels that have no center, such as the
$2 \times 2$ kernel of the next figure and the $1 \times 2$ edge detector.

![Two-dimensional cross-correlation. The shaded portions are the first output element and the input and kernel elements used to compute it: $0\times0+1\times1+3\times2+4\times3=19$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/correlation.svg)

In the figure the input has shape $3 \times 3$ and the kernel
has shape $2 \times 2$. The window starts at the upper-left corner of the input
and slides one element at a time, from left to right and from top to bottom.
The window fits at four positions, which give the four outputs

$$
\begin{aligned}
0\times0+1\times1+3\times2+4\times3 &= 19,\\
1\times0+2\times1+4\times2+5\times3 &= 25,\\
3\times0+4\times1+6\times2+7\times3 &= 37,\\
4\times0+5\times1+7\times2+8\times3 &= 43.
\end{aligned}
$$

Note that the output is smaller than the input, since the window must lie
inside the input. Along each axis it fits at $n - k + 1$ positions, so the
output has shape

$$(n_\textrm{h}-k_\textrm{h}+1) \times (n_\textrm{w}-k_\textrm{w}+1).$$

The $3 \times 3$ input with a $2 \times 2$ kernel gives $2 \times 2$, the
$6 \times 8$ edge image with a $1 \times 2$ kernel gives $6 \times 7$, and a
$28 \times 28$ Fashion-MNIST image with a $5 \times 5$ kernel gives
$24 \times 24$. If the shrinking is unwelcome, a border of zeros around the
input, called *padding*, undoes it; that section treats it together
with other ways to control the output shape.

### Implementation

The function `corr2d` computes the equation in the plainest way
possible, with two loops over the output positions.

In [ ]:
def corr2d(X, K):
    """Compute 2D cross-correlation."""
    h, w = K.shape
    Y = jnp.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y = Y.at[i, j].set((X[i:i + h, j:j + w] * K).sum())
    return Y

Does `corr2d` reproduce the four outputs of the figure? We
construct its input and kernel and compare.

In [ ]:
X = d2l.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = d2l.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

The output matches the figure, number for number.

### Convolution and Cross-Correlation

Strictly speaking, `corr2d` does not compute a convolution. The mathematical
*convolution* flips the kernel horizontally and vertically before sliding it
(that section). The numbers show how little is at stake. For the
input of the figure, a layer that computes true convolutions
produces the same outputs 19, 25, 37, and 43 with the kernel
$\mathbf{K}' = \begin{bmatrix} 3 & 2 \\ 1 & 0 \end{bmatrix}$, and since
kernels are learned from data, it would simply learn $\mathbf{K}'$ in place
of $\mathbf{K}$. From now on we say convolution for both, as the libraries
do.


## Convolutional Layers

### The Layer

A convolutional layer cross-correlates its input with a kernel and adds a
scalar bias. Its two parameters are the kernel and the bias. The kernel
starts from random values, drawn from $\mathcal{N}(0, 0.01^2)$ like the
weights of the MLP in that section, and the bias starts at
zero. The class `Conv2D` declares the two parameters in its
constructor, and its forward method calls `corr2d` and adds the bias. A layer
whose kernel has height $h$ and width $w$ is called an $h \times w$
convolutional layer.

In [ ]:
class Conv2D(nnx.Module):
    def __init__(self, kernel_size, rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.weight = nnx.Param(
            nnx.initializers.normal(stddev=0.01)(rngs.params(), kernel_size))
        self.bias = nnx.Param(jnp.zeros(1))

    def __call__(self, x):
        return corr2d(x, self.weight) + self.bias

Unlike a fully connected layer, this layer is not tied to one input size,
since its parameters belong to the kernel alone. One $2 \times 2$ layer with
its five parameters applies to the $3 \times 3$ input `X` and to a
$6 \times 8$ image alike, and returns outputs of shape $2 \times 2$ and
$5 \times 7$.

In [ ]:
layer = Conv2D((2, 2))
layer(X).shape, layer(d2l.ones((6, 8))).shape

### Edge Detection

It is time to run the edge detector of the opening. The image has
$6 \times 8$ pixels; its middle four columns are black (0) and the rest is
white (1).

In [ ]:
X = jnp.ones((6, 8))
X = X.at[:, 2:6].set(0)
X

The kernel has height 1 and width 2:

$$\mathbf{K} = \begin{bmatrix} 1 & -1 \end{bmatrix}.$$

At position $(i, j)$ it computes
$[\mathbf{X}]_{i,j} - [\mathbf{X}]_{i,j+1}$, the difference between two
horizontally adjacent pixels, which is 0 wherever they are equal. This is a
*finite difference*: up to sign, it approximates the derivative of the
brightness along the row. A derivative vanishes where nothing changes, which
is why the kernel responds only where the brightness does.

In [ ]:
K = d2l.tensor([[1.0, -1.0]])

Cross-correlating `X` with `K` gives the edge map `Y`.

In [ ]:
Y = corr2d(X, K)
Y

The output is 1 in the column where white turns to black, $-1$ in the column
where black turns to white, and 0 everywhere else. The same kernel applied to
the transposed image, whose band is horizontal, should find nothing.

In [ ]:
corr2d(d2l.transpose(X), K)

The output is zero everywhere: $(1, -1)$ detects only vertical edges. Edges in
other directions need other kernels, and so do corners, textures, and every
other pattern a network may need.

### Learning a Kernel

We cannot design all of these by hand, least of all for the intermediate
layers of a stack, where we do not even know which patterns to ask for. Can
gradient descent find a kernel from the input `X` and the desired output `Y`
alone? Let us try it on the one kernel whose answer we know. We start from a
small random $1 \times 2$ kernel, drawn as before, measure the squared error
between the layer's output and `Y`, summed over all 42 output positions, and
take 10 gradient steps with learning rate 0.03. Instead of `Conv2D` we use
the built-in convolutional layer, which computes the same cross-correlation
fast. It expects two more axes, one for the examples of a minibatch and one
for channels (that section), so we reshape `X` and `Y`
accordingly. We omit the bias, which the edge map does not need.

In [ ]:
# padding='VALID' evaluates only where the window fits, as corr2d does
conv2d = nnx.Conv(1, 1, kernel_size=(1, 2), use_bias=False, padding='VALID',
                  kernel_init=nnx.initializers.normal(stddev=0.01),
                  rngs=nnx.Rngs(d2l.get_key()))
# Layers take (example, height, width, channel): one example, one channel
X = X.reshape((1, 6, 8, 1))
Y = Y.reshape((1, 6, 7, 1))
lr = 3e-2  # Learning rate

def loss(model, X, Y):
    return ((model(X) - Y) ** 2).sum()

for i in range(10):
    l, grads = nnx.value_and_grad(loss)(conv2d, X, Y)
    conv2d.kernel[...] -= lr * grads.kernel[...]
    if i % 3 == 0:
        print(f'step {i}, loss {l:.3f}')

Before the first step the loss is close to 12: the small random kernel
outputs nearly 0 everywhere, so each of the 12 edge positions misses its
target $\pm 1$ by about 1. After nine steps the loss is below 0.01. The
learned kernel is close to $(1, -1)$:

In [ ]:
conv2d.kernel[...].reshape((1, 2))

Both weights are within about 0.02 of the hand-designed detector. Nothing in
the training procedure was specific to edges: given the outputs it should
produce, gradient descent finds the kernel. Admittedly, we made its task easy
by supplying the edge map. In a deep network nobody supplies the intermediate
outputs, and the kernels of every layer are learned from the loss at the end
of the network.


## Convolution as Matrix Multiplication

The two loops of `corr2d` are easy to read and far too slow for real images
and networks, since they visit one output position at a time. A standard
remedy is to turn the whole computation into one large matrix product, the
operation that GPUs execute best. Every output element in
the figure is a dot product: the kernel, flattened into a
vector of length $k_\textrm{h} k_\textrm{w}$, times the input patch under
the window, flattened the same way. If we extract each
of the $(n_\textrm{h}-k_\textrm{h}+1)(n_\textrm{w}-k_\textrm{w}+1)$ patches
that the window visits, flatten it into a row, and stack the rows, we obtain a
matrix with one row per output position. The whole cross-correlation then
becomes a single product of this patch matrix with the flattened kernel. The
rearrangement is called *im2col*, since it turns image patches into the
columns (here rows) of a matrix.

The patch matrix takes memory, since each input element is copied into up
to $k_\textrm{h} k_\textrm{w}$ rows. We build it for the input and kernel
of the figure and check that one matrix product reproduces
`corr2d`.

In [ ]:
X = d2l.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = d2l.tensor([[0.0, 1.0], [2.0, 3.0]])
h, w = K.shape
h_o, w_o = X.shape[0] - h + 1, X.shape[1] - w + 1
patches = d2l.stack([d2l.reshape(X[i:i + h, j:j + w], (-1,))
                     for i in range(h_o) for j in range(w_o)])
Y_mat = d2l.reshape(d2l.matmul(patches, d2l.reshape(K, (-1, 1))), (h_o, w_o))
Y_mat, d2l.reduce_sum(d2l.abs(Y_mat - corr2d(X, K)))

The sum of absolute differences is zero: the single matrix product and the
loops of `corr2d` compute the same output.

We could equally well unroll the kernel instead of the input. The result is
a sparse, banded matrix $\mathbf{W}$ that multiplies the flattened image, and
we have met it before: it is the two-dimensional counterpart of panel (c) of
the figure, in which every row holds the same few
weights. A convolution is a fully connected layer whose weight matrix has
this pattern, which is where the derivation of that section
started. The exercises construct $\mathbf{W}$ explicitly.

## Feature Maps and Receptive Fields

A single layer sees only a window the size of its kernel. Recognizing an
object requires seeing much more of the image, so we need a way to measure
how much of the input a unit sees. The output of a convolutional layer is a
feature map (that section); the edge map `Y` is one,
for vertical edges. For an element $x$ of a feature map, its *receptive
field* is the set of input pixels that can affect $x$ during forward
propagation. Relative to an earlier layer, it is the set of that layer's
elements that can affect $x$. The term comes from neurophysiology, where it
names the part of the visual field to which a neuron responds
(the figure).

### Stacked Layers

In the figure the receptive field of the shaded output 19 is
the four shaded input elements. Call the $2 \times 2$ output $\mathbf{Y}$, and
add a second $2 \times 2$ convolutional layer that takes $\mathbf{Y}$ as input
and outputs a single element $z$. The receptive field of $z$ on $\mathbf{Y}$
is all four elements of $\mathbf{Y}$, and on the input it is all nine input
elements. Stacking layers lets an element respond to a larger region of the
input.

How fast does the region grow? A layer with a $k \times k$ kernel reaches
$k - 1$ positions beyond a single position of its input, so each layer widens
the receptive field by $k - 1$. Starting from one pixel, an element at the top
of a stack of $L$ layers with kernel sizes $k_1, \ldots, k_L$ has a receptive
field of side length

$$r = 1 + \sum_{i=1}^{L} (k_i - 1).$$

For $L$ stacked $3 \times 3$ layers each layer adds 2, so the stack sees
$(2L + 1) \times (2L + 1)$ input pixels: two layers cover $5 \times 5$ and
three cover $7 \times 7$ (the figure). This slow,
linear growth is no reason to prefer large kernels. Three $3 \times 3$
layers match the receptive field of one $7 \times 7$ layer with fewer
parameters, $27$ weights instead of $49$ per pair of input and output
channels, and with three nonlinearities instead of one.

![Receptive field of one unit after three one-dimensional layers with kernel size 3. Shaded cells are the units that the top unit depends on, and the lines connect each of them to its inputs. Each layer widens the receptive field by 2, to 7 inputs.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-receptive-field.svg)

### Effective Receptive Fields

The count $r$ is generous. It gives the theoretical receptive field, the set
of inputs that can affect an activation at all, and says nothing about how
much each of them does. @Luo.Li.Urtasun.ea.2016 measured the
*effective* receptive field through gradients: in both randomly initialized
and trained networks it concentrates near the center of the theoretical
region, with a roughly Gaussian profile. A unit at the top of a deep stack
can see far, but it looks mostly at the middle.

Two defects of the layer are left to repair. Every layer shrinks the image
and uses the pixels near its border less often than those in the middle, and
a single-channel kernel detects only one pattern.
that section takes up the first and that section the
second.

## Summary

A convolutional layer cross-correlates its input with a small kernel and adds
a bias: every output element is the sum of the elementwise product of the
kernel with one window of the input, and the window visits every position
where it fits. The same two weights $(1, -1)$ then find a vertical edge
wherever it occurs, and gradient descent recovers these weights from an image
and its edge map, as it learns the kernels of every layer of a CNN from the
loss. Mathematical convolution flips the kernel, which makes no difference
when the kernel is learned. The operation can be written as one matrix
product, which accelerators execute efficiently. Stacking layers
enlarges the region of the input that each unit sees, by $k - 1$ pixels per
layer with a $k \times k$ kernel.

## Exercises

1. [code] **Diagonal edges.** Construct a $6 \times 6$ image `X` whose
   entries are $1$ on and below the main diagonal and $0$ above it. Let
   $\mathbf{K}$ be the kernel of the equation.
    1. Apply $\mathbf{K}$ and $\mathbf{K}^\top$ to `X` with `corr2d`.
       Give the positions and signs of the nonzero entries of each
       output, and explain them from the pixel differences that each
       kernel computes.
    1. Let $\mathbf{A}$ be any input and $\mathbf{B}$ any kernel. Show
       that cross-correlating $\mathbf{A}^\top$ with $\mathbf{B}$ gives
       the transpose of cross-correlating $\mathbf{A}$ with
       $\mathbf{B}^\top$. What does this identity predict for `corr2d`
       applied to the transpose of `X` with $\mathbf{K}$?

1. [code] **Hand-designed kernels.** Design some kernels manually.
    1. Given a directional vector $\mathbf{v} = (v_1, v_2)$, derive an
       edge-detection kernel that detects edges orthogonal to $\mathbf{v}$,
       i.e., edges in the direction $(v_2, -v_1)$.
    1. Derive a finite difference operator for the second derivative.
       Which structures in images respond most strongly to it?
    1. What is the minimum size of a kernel to obtain a derivative of
       order $d$?
    1. How would you design a blur kernel? Why might you want to use such
       a kernel?
    1. Load a photograph, convert it to one channel, and downsample it to
       about $100 \times 100$ pixels (`corr2d` loops in Python). Apply the
       edge detector, the blur kernel, and the sharpening kernel "twice
       the identity minus the blur", and display each result next to the
       original. What does the edge map of a photograph show that the
       edge map of the black band could not?

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2024/assignment2/),
    ConvolutionalNetworks notebook.*

1. [code] **Convolution as a banded matrix.** The patch matrix of
   that section unfolds the input. The opposite
   construction unrolls the kernel. Let $\mathbf{X}$ be a $4 \times 4$
   input, $\mathbf{K}$ a $3 \times 3$ kernel, and $\textrm{vec}(\cdot)$
   the row-by-row flattening.
    1. Construct the $4 \times 16$ matrix $\mathbf{W}$ with
       $\mathbf{W}\,\textrm{vec}(\mathbf{X}) = \textrm{vec}(\mathbf{Y})$,
       where $\mathbf{Y}$ is the cross-correlation of $\mathbf{X}$ with
       $\mathbf{K}$. How many nonzero entries does each row have, and how
       many distinct values appear in $\mathbf{W}$?
    1. Build $\mathbf{W}$ in code from `K` and confirm that the product
       reproduces `corr2d(X, K)`.
    1. For an $n \times n$ input and a $k \times k$ kernel, count the
       entries of $\mathbf{W}$ stored as a dense array, the nonzero
       entries of $\mathbf{W}$, and the entries of the patch matrix. How
       does each count grow with $n$?
    1. Keep the sparsity pattern of $\mathbf{W}$ but give every row its
       own $k^2$ values. The result is a locally connected layer: the
       fourth-order tensor $\mathsf{V}$ of the equation
       restricted to a $k \times k$ window of offsets $(a, b)$, before
       translation equivariance ties the weights. For an $n \times n$
       input, count the parameters of this layer, of a fully connected
       layer with the same input and output shapes, and of the
       convolution.

    *Adapted from CMU 11-785,
    [Homework 2 Part 1](https://deeplearning.cs.cmu.edu/S21/index.html).*

1. [code] **Effective receptive field.** ● The theoretical receptive
   field $r$ of this section bounds which inputs *can* influence an
   output; the effective receptive field measures how much
   they actually do [@Luo.Li.Urtasun.ea.2016]. Build a stack of $L$
   single-channel $3 \times 3$ convolutional layers with no
   padding, no bias, and no nonlinearity on a $32 \times 32$ input, using
   the built-in layer of that section. Draw every
   kernel entry independently from a standard normal distribution.
    1. For $L = 3$ and a single draw of the kernels, backpropagate a unit
       gradient from an output unit near the center of the last layer to
       the input and plot the magnitude of the input gradient. Compare its
       support with the $7 \times 7$ theoretical receptive field, and say
       where the magnitude is largest.
    1. Average the squared input gradient over 500 draws of the kernels,
       for $L = 3$ and for $L = 10$. Sum each averaged map over its rows
       and normalize the resulting column profile to sum to one. How does
       the ratio of the profile's standard deviation to the theoretical
       side length $2L + 1$ change with depth?
    1. Show that the expected squared input gradient, summed over rows,
       is proportional to the convolution of $L$ copies of the vector
       $(1, 1, 1)$. Derive how the standard deviation of the normalized
       profile grows with $L$.

1. [code] **Backward pass of a convolution.** ● Let $\mathbf{Y}$ be the
   single-channel cross-correlation of an input $\mathbf{X}$ with a kernel
   $\mathbf{K}$, as computed by `corr2d`, and let $\ell$ be a scalar loss
   that depends on $\mathbf{X}$ and $\mathbf{K}$ only through $\mathbf{Y}$.
    1. Derive $\partial \ell/\partial \mathbf{K}$ and
       $\partial \ell/\partial \mathbf{X}$ in terms of
       $\partial \ell/\partial \mathbf{Y}$, in the style of the
       backpropagation equations of that section.
    1. Implement your expression for $\partial \ell/\partial \mathbf{K}$
       without automatic differentiation for the loss
       $\ell = \sum_{i,j} G_{ij} Y_{ij}$, where $\mathbf{G}$ is a fixed
       random matrix of the shape of $\mathbf{Y}$. Compare it with a
       centered finite-difference gradient with step size $10^{-2}$ on a
       small random `X` and `K`, and report the largest relative error.
    1. Check your expression for $\partial \ell/\partial \mathbf{K}$
       against the kernel gradient that automatic differentiation
       computes in the experiment of that section.

    *Adapted from Michael Nielsen,
    [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/chap6.html),
    Chapter 6, and Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2024/assignment2/),
    ConvolutionalNetworks notebook.*

[Discussions](https://d2l.discourse.group/t/17996)